# 26 - Continuous-Control RL With CEM

## What / Why / How

**What we are trying to do:** Optimize a continuous action sequence with the cross-entropy method.

**Why this matters:** This bridges trajectory optimization, MPC, and reinforcement learning without requiring a deep RL framework.

**How we will do it:** Sample candidate action sequences, keep elites, update the sampling distribution, and roll out the best sequence.

## Goal

Learn a simple continuous-control optimizer: the cross-entropy method.

This is a useful bridge between planning, model predictive control, and reinforcement learning.

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see plots: pip install -r requirements.txt")

In [ ]:
rng = np.random.default_rng(26)
horizon = 35
dt = 0.08
target = np.array([1.0, 0.0])

def evaluate_sequence(actions):
    x = np.array([-1.0, 0.0])
    v = np.array([0.0, 0.0])
    cost = 0.0
    for a in actions.reshape(horizon, 2):
        a = np.clip(a, -2, 2)
        v = 0.95 * v + a * dt
        x = x + v * dt
        cost += np.linalg.norm(x - target) ** 2 + 0.02 * np.linalg.norm(a) ** 2
    cost += 20 * np.linalg.norm(x - target) ** 2
    return cost

dim = horizon * 2
mean = np.zeros(dim)
std = np.ones(dim) * 0.8
history = []
for iteration in range(18):
    candidates = rng.normal(mean, std, size=(300, dim))
    costs = np.array([evaluate_sequence(c) for c in candidates])
    elite = candidates[np.argsort(costs)[:30]]
    mean = elite.mean(axis=0)
    std = elite.std(axis=0) + 1e-3
    history.append(costs.min())
print("best cost:", history[-1])

In [ ]:
actions = mean.reshape(horizon, 2)
x = np.array([-1.0, 0.0])
v = np.array([0.0, 0.0])
route = [x.copy()]
for a in actions:
    v = 0.95 * v + np.clip(a, -2, 2) * dt
    x = x + v * dt
    route.append(x.copy())
route = np.array(route)
print("final position:", route[-1], "error:", np.linalg.norm(route[-1] - target))

In [ ]:
if HAS_PLOT:
    plt.figure(figsize=(5, 4))
    plt.plot(route[:, 0], route[:, 1], marker="o", markersize=2)
    plt.scatter([target[0]], [target[1]], c="tab:red")
    plt.axis("equal")
    plt.grid(True, alpha=0.3)
    plt.title("CEM optimized action sequence")
    plt.show()
else:
    plot_unavailable()

## Exercises

1. Add an obstacle cost.
2. Replan every 5 steps like MPC.
3. Compare this with Q-learning from notebook 8.